# Transformer Architecture — AI Lab Instructor Notebook

*NLP & Transformers · Hard*



10 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Transformer Architecture — Student Lab

Implement a minimal Transformer encoder-style block forward pass in NumPy:
- embeddings + sinusoidal positional encoding
- multi-head self-attention (from scratch)
- FFN
- residuals + layer norm (Pre-LN)

Focus: shapes, masking, and correctness.

## Section 0 — Utilities (softmax, LayerNorm, activation)

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # Transformer Architecture — Instructor Lab

Complete forward-pass implementations + short rationales. Offline, deterministic.

## Utilities
Rationale: stable softmax + layer norm are foundational; most bugs are axis/shape mistakes.

## Task 2: Section 0 — Utilities (softmax, LayerNorm, activation)
*Section: Utilities (softmax, LayerNorm, activation)*

## Section 0 — Utilities (softmax, LayerNorm, activation)

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    m = np.max(x, axis=axis, keepdims=True)
    z = x - m
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)

def layer_norm(x: np.ndarray, gamma: np.ndarray, beta: np.ndarray, eps: float = 1e-5) -> np.ndarray:
    mu = np.mean(x, axis=-1, keepdims=True)
    var = np.mean((x - mu) ** 2, axis=-1, keepdims=True)
    xhat = (x - mu) / np.sqrt(var + eps)
    return xhat * gamma + beta

def gelu(x: np.ndarray) -> np.ndarray:
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * (x ** 3))))

**Why this works.** ## Utilities
Rationale: stable softmax + layer norm are foundational; most bugs are axis/shape mistakes.

## Task 3: Implement sinusoidal positional encoding `pos_enc(T, D)` returning (T,D).
*Section: Embeddings + positional encoding*

## Section 1 — Embeddings + positional encoding

### Task 1.1
Implement sinusoidal positional encoding `pos_enc(T, D)` returning (T,D).
Use the standard formula with sin/cos on even/odd dims.

`PE(pos, 2i)   = sin(pos / 10000^(2i / D))`
`PE(pos, 2i+1) = cos(pos / 10000^(2i / D))`

Here `pos` is the token position, `i` is the dimension-pair index, and `D` is the embedding size.

In [ ]:
def pos_enc(T: int, D: int) -> np.ndarray:
    pe = np.zeros((T, D), dtype=np.float64)
    pos = np.arange(T)[:, None]
    i = np.arange(D)[None, :]
    denom = np.power(10000.0, (2 * (i // 2)) / D)
    angles = pos / denom
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe

T, Dm, V = 6, 12, 20
E = 0.02 * rng.standard_normal((V, Dm))
tokens = rng.integers(0, V, size=(2, T), dtype=np.int64)
X = E[tokens] + pos_enc(T, Dm)[None, :, :]
check('X_shape', X.shape == (2, T, Dm))

In [ ]:
# Checks
check('X_shape', X.shape == (2, T, Dm))

**Why this works.** ## Embeddings + positional encodings
Rationale: attention itself is permutation-invariant; positional information breaks symmetry.

## Task 4: Implement token embedding lookup.
*Section: Embeddings + positional encoding*

### Task 1.2
Implement token embedding lookup.
Given `tokens` (B,T) and embedding table `E` (V,D), return `X` (B,T,D).

In [ ]:
def embed(tokens: np.ndarray, E: np.ndarray) -> np.ndarray:
    return E[tokens]

B, T, V, Dm = 2, 6, 20, 12
E = 0.02 * rng.standard_normal((V, Dm))
tokens = rng.integers(0, V, size=(B, T), dtype=np.int64)
X = embed(tokens, E)
check('embed_shape', X.shape == (B, T, Dm))

In [ ]:
# Checks
check('embed_shape', X.shape == (B, T, Dm))

**Why this works.** ## Embeddings + positional encodings
Rationale: attention itself is permutation-invariant; positional information breaks symmetry.

## Task 5: Implement masks:
*Section: Multi-head self-attention*

## Section 2 — Multi-head self-attention

### Task 2.1
Implement masks:
- `causal_mask(T)` -> (1, 1, T, T)
- `padding_mask(lengths, T)` -> (B, 1, 1, T)
Mask convention: 1=keep, 0=mask.

In [ ]:
def causal_mask(T: int) -> np.ndarray:
    return np.tril(np.ones((T, T), dtype=np.int64))[None, None, :, :]

def padding_mask(lengths: np.ndarray, T: int) -> np.ndarray:
    idx = np.arange(T)[None, :]
    m = (idx < lengths[:, None]).astype(np.int64)
    return m[:, None, None, :]

check('cm_shape', causal_mask(4).shape == (1,1,4,4))
check('pm_shape', padding_mask(np.array([2,4]), 4).shape == (2,1,1,4))

In [ ]:
# Checks
check('cm_shape', causal_mask(4).shape == (1,1,4,4))
check('pm_shape', padding_mask(np.array([2,4]), 4).shape == (2,1,1,4))

**Why this works.** ## Multi-head self-attention
Rationale: do attention per head, then concatenate. Masks are applied before softmax.

## Task 6: Implement head split/merge utilities.
*Section: Multi-head self-attention*

### Task 2.2
Implement head split/merge utilities.

In [ ]:
def split_heads(x: np.ndarray, n_heads: int) -> np.ndarray:
    B, T, Dm = x.shape
    Dh = Dm // n_heads
    return x.reshape(B, T, n_heads, Dh).transpose(0, 2, 1, 3)

def merge_heads(xh: np.ndarray) -> np.ndarray:
    B, h, T, Dh = xh.shape
    return xh.transpose(0, 2, 1, 3).reshape(B, T, h * Dh)

x = rng.standard_normal((2, 5, 12))
xh = split_heads(x, 3)
x2 = merge_heads(xh)
check('split_shape', xh.shape == (2,3,5,4))
check('merge_roundtrip', float(np.max(np.abs(x2 - x))) < 1e-12)

In [ ]:
# Checks
check('split_shape', xh.shape == (2,3,5,4))
check('merge_roundtrip', float(np.max(np.abs(x2 - x))) < 1e-12)

**Why this works.** ## Multi-head self-attention
Rationale: do attention per head, then concatenate. Masks are applied before softmax.

## Task 7: Implement scaled dot-product attention over heads.
*Section: Multi-head self-attention*

### Task 2.3
Implement scaled dot-product attention over heads.
Inputs: Q,K,V each shape (B,h,T,Dh).
Mask shape should be broadcastable to (B,h,T,T).
Return (O,A) where O=(B,h,T,Dh), A=(B,h,T,T).

In [ ]:
def attn_heads(Q: np.ndarray, K: np.ndarray, V: np.ndarray, mask=None):
    Dh = Q.shape[-1]
    scores = (Q @ np.swapaxes(K, -1, -2)) / np.sqrt(Dh)
    if mask is not None:
        scores = np.where(mask.astype(bool), scores, -1e9)
    A = softmax(scores, axis=-1)
    O = A @ V
    return O, A

B, h, T, Dh = 2, 3, 7, 4
Q = rng.standard_normal((B, h, T, Dh))
K = rng.standard_normal((B, h, T, Dh))
V = rng.standard_normal((B, h, T, Dh))
mask = np.tril(np.ones((T, T), dtype=np.int64))[None, None, :, :]

O, A = attn_heads(Q, K, V, mask=mask)
check('O_shape', O.shape == (B,h,T,Dh))
check('A_shape', A.shape == (B,h,T,T))
check('A_rowsum1', float(np.max(np.abs(A.sum(axis=-1) - 1.0))) < 1e-8)

In [ ]:
# Checks
check('O_shape', O.shape == (B,h,T,Dh))
check('A_shape', A.shape == (B,h,T,T))
check('A_rowsum1', float(np.max(np.abs(A.sum(axis=-1) - 1.0))) < 1e-8)

**Why this works.** ## Multi-head self-attention
Rationale: do attention per head, then concatenate. Masks are applied before softmax.

## Task 8: Implement a multi-head self-attention forward pass:
*Section: Multi-head self-attention*

### Task 2.4
Implement a multi-head self-attention forward pass:
- project Q,K,V with `Wq,Wk,Wv`
- split into heads
- apply attention
- merge heads
- output projection `Wo`
Return (Y, A).

In [ ]:
def mha_forward(X: np.ndarray, params: dict, n_heads: int, mask=None):
    Wq, Wk, Wv, Wo = params['Wq'], params['Wk'], params['Wv'], params['Wo']
    Q = split_heads(X @ Wq, n_heads)
    K = split_heads(X @ Wk, n_heads)
    V = split_heads(X @ Wv, n_heads)
    O, A = attn_heads(Q, K, V, mask=mask)
    Om = merge_heads(O)
    Y = Om @ Wo
    return Y, A

B, T, Dm = 2, 7, 12
h = 3
X = rng.standard_normal((B, T, Dm))
params = {
  'Wq': 0.02 * rng.standard_normal((Dm, Dm)),
  'Wk': 0.02 * rng.standard_normal((Dm, Dm)),
  'Wv': 0.02 * rng.standard_normal((Dm, Dm)),
  'Wo': 0.02 * rng.standard_normal((Dm, Dm)),
}
Y, A = mha_forward(X, params, n_heads=h, mask=causal_mask(T))
check('Y_shape', Y.shape == (B, T, Dm))
check('A_shape2', A.shape == (B, h, T, T))

In [ ]:
# Checks
check('Y_shape', Y.shape == (B, T, Dm))
check('A_shape2', A.shape == (B, h, T, T))

**Why this works.** ## Multi-head self-attention
Rationale: do attention per head, then concatenate. Masks are applied before softmax.

## Task 9: Implement FFN forward: `FFN(x) = (gelu(x W1 + b1)) W2 + b2`.
*Section: Feed-forward network (FFN)*

## Section 3 — Feed-forward network (FFN)

### Task 3.1
Implement FFN forward: `FFN(x) = (gelu(x W1 + b1)) W2 + b2`.

In [ ]:
def ffn_forward(X: np.ndarray, params: dict):
    W1, b1, W2, b2 = params['W1'], params['b1'], params['W2'], params['b2']
    H = gelu(X @ W1 + b1)
    return H @ W2 + b2

Dm = 12
params_ffn = {
  'W1': 0.02 * rng.standard_normal((Dm, 4*Dm)),
  'b1': np.zeros((4*Dm,), dtype=np.float64),
  'W2': 0.02 * rng.standard_normal((4*Dm, Dm)),
  'b2': np.zeros((Dm,), dtype=np.float64),
}
x = rng.standard_normal((2, 7, Dm))
y = ffn_forward(x, params_ffn)
check('ffn_shape', y.shape == x.shape)

In [ ]:
# Checks
check('ffn_shape', Y.shape == X.shape)

**Why this works.** ## FFN
Rationale: provides per-token nonlinearity and feature mixing independent of attention.

## Task 10: Implement `transformer_block_forward(x, params, n_heads, attn_mask=None)` returning (y, attn_weights).
*Section: Transformer block (Pre-LN)*

## Section 4 — Transformer block (Pre-LN)

Pre-LN block:
1) `x1 = x + MHA(LN(x))`
2) `x2 = x1 + FFN(LN(x1))`

Here `x` should already contain token embeddings plus positional encodings.

### Task 4.1
Implement `transformer_block_forward(x, params, n_heads, attn_mask=None)` returning (y, attn_weights).

In [ ]:
$$10

In [ ]:
# Checks
check('y_shape', y.shape == x.shape)
check('A_shape', A.shape == (B, h, T, T))
check('A_rowsum', float(np.max(np.abs(A.sum(axis=-1) - 1.0))) < 1e-8)

**Why this works.** ## Transformer block (Pre-LN)
Rationale: Pre-LN improves gradient flow and training stability in deep stacks.